# 02 — Deep Blue data: organ and lesion kinetics, fits and TIA

Input: the NIfTI files written by notebook 01 (`data/processed/deepblue/`), physical activity in Bq/mL on the SPECT grid of each scan, masks drawn on each scan, MIM TIA map registered to scan 1.

These are the two cases of the SNMMI Lu-177 Dosimetry Challenge. In the challenge papers our **patient 4 is "patient A"** (7.21 GBq, scans at 3.7, 27.7, 103.1, 124.0 h) and **patient 6 is "patient B"** (7.31 GBq, scans at 3.7, 32.6, 99.6, 193.3 h). The fitting step was analysed in Kurkowska et al., *J Nucl Med* 2025;66:441–448 (Part 4 of the challenge series), which is the reference for the comparison at the end of this notebook.

**Steps**
1. Liver: map every scan onto scan 1 with a rigid registration around the liver and use the **scan-1 liver mask as reference** at all time points; healthy liver = scan-1 liver minus lesions.
2. Visual check of lesion 2 of patient 6 before fitting.
3. Time-activity curves (TACs) of organs and lesions.
4. Fits with the uptake–washout model for all structures (mono-exponential and AIC only as checks), TIA per structure, uncertainty by Monte Carlo on the data points.
5. Extrapolation after the last scan as a limitation of the protocol.
6. Comparison with the MIM TIA map and with the SNMMI challenge findings.

**About this notebook.** Organ and lesion time-activity curves, uptake-washout fits, TIA with Monte Carlo uncertainty.
- **Inputs:** nb01 outputs.
- **Outputs:** `data/processed/deepblue/kinetics/` (`tac.csv`, fits, TIA, uncertainty).
- **Run time:** minutes.
- **Environment:** .venv.

In [ ]:
import json
import sys
from pathlib import Path

import lmfit
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import SimpleITK as sitk

WS = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "paths.py").exists())
sys.path.insert(0, str(WS / "src"))
from paths import DATASET
PROC = WS / "data" / "processed" / "deepblue"
OUT = PROC / "kinetics"
PATIENTS = ["patient_4", "patient_6"]
SNMMI_LABEL = {"patient_4": "A", "patient_6": "B"}
OUT.mkdir(parents=True, exist_ok=True)

## Loading

One entry per scan, ordered by time after injection. Mask files are named `organs_<name>` and `lesions_<name>`.

In [ ]:
def read(path):
    return sitk.ReadImage(str(path))


def arr(image):
    return sitk.GetArrayFromImage(image)


def voxel_ml(image):
    return float(np.prod(image.GetSpacing())) / 1000.0


def activity_mbq(image, mask):
    return float(arr(image)[arr(mask) > 0].sum()) * voxel_ml(image) / 1e6


def centroid(mask):
    idx = np.argwhere(arr(mask) > 0).mean(axis=0)[::-1]
    return np.array(mask.TransformContinuousIndexToPhysicalPoint(idx.tolist()))


def dice(a, b):
    a, b = arr(a) > 0, arr(b) > 0
    return 2.0 * (a & b).sum() / (a.sum() + b.sum())


def mask_like(array, reference):
    image = sitk.GetImageFromArray(array.astype(np.uint8))
    image.CopyInformation(reference)
    return image


data = {}
for patient in PATIENTS:
    pdir = PROC / patient
    timing = json.loads((pdir / "timing.json").read_text())
    scans = {}
    for scan, t_h in sorted(timing["t_h"].items(), key=lambda kv: kv[1]):
        sdir = pdir / scan
        scans[scan] = {
            "t_h": t_h,
            "spect": read(sdir / "spect_bqml.nii.gz"),
            "ct": read(sdir / "ct_on_spect.nii.gz"),
            "masks": {p.name.removesuffix(".nii.gz"): read(p) for p in sorted((sdir / "masks").glob("*.nii.gz"))},
        }
    data[patient] = {"timing": timing, "scans": scans, "tia": read(pdir / "tia_mim_bqml_s.nii.gz")}
    print(patient, {s: round(v["t_h"], 1) for s, v in scans.items()}, list(scans["scan1"]["masks"]))

half_lives = {data[p]["timing"]["half_life_s"] for p in PATIENTS}
assert len(half_lives) == 1
LAM_PHYS = np.log(2) / (half_lives.pop() / 3600.0)
print(f"lambda_phys = {LAM_PHYS:.5f} 1/h, T1/2 = {np.log(2) / LAM_PHYS:.1f} h")

## Liver: scan-1 masks as reference

The liver contours drawn on the four scans are not consistent: the liver volume changes by 10–13% between scans, and for patient 6 the overlap between the liver mask and lesions 3–4 changes from scan to scan (lesion 4 is 0% inside the liver at scan 1 and 59% at scan 3). The liver is therefore delineated once, on scan 1, and the same contour is used at all time points.

The patient was repositioned between scans (the liver moves by up to ~17 cm along z in scanner coordinates), so the scan-1 mask cannot be applied directly to the other scans. Each scan is aligned to scan 1 with a **rigid CT-to-CT registration restricted to the liver region** (scan-1 liver mask dilated by 16 mm), which is the same idea as the contour-based alignment used by MIM for the TIA map:

- initial translation from the centroids of the liver masks of the two scans;
- Mattes mutual information, multi-resolution (shrink 4, 2, 1), regular-step gradient descent;
- the SPECT of scan $k$ is then resampled on the scan-1 grid (linear interpolation, which preserves the total activity to within a fraction of a percent).

Quality control:
- Dice coefficient between the scan-1 liver mask and the liver mask drawn on scan $k$, mapped to scan 1, before (centroid alignment only) and after registration. The two contours were drawn independently, so values around 0.85–0.9 are the practical ceiling;
- coverage: fraction of the scan-1 liver that falls inside the field of view of scan $k$.

A rigid transform cannot model liver deformation (breathing, different filling of the bowel); this is a limitation shared with the challenge reference workflow.

**Healthy liver** is the scan-1 liver minus all scan-1 lesion masks. The SNMMI challenge asked for healthy-liver doses and identified the inclusion of hepatic lesions as the main source of liver variability.

In [ ]:
def register_liver(fixed_ct, moving_ct, fixed_liver, moving_liver, seed=0):
    init = sitk.Euler3DTransform()
    init.SetCenter(centroid(fixed_liver).tolist())
    init.SetTranslation((centroid(moving_liver) - centroid(fixed_liver)).tolist())
    reg = sitk.ImageRegistrationMethod()
    reg.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)
    reg.SetMetricFixedMask(sitk.BinaryDilate(fixed_liver, [8, 8, 8]))
    reg.SetMetricSamplingStrategy(reg.RANDOM)
    reg.SetMetricSamplingPercentage(0.25, seed)
    reg.SetInterpolator(sitk.sitkLinear)
    reg.SetOptimizerAsRegularStepGradientDescent(learningRate=2.0, minStep=1e-3, numberOfIterations=300,
                                                 relaxationFactor=0.5)
    reg.SetOptimizerScalesFromPhysicalShift()
    reg.SetShrinkFactorsPerLevel([4, 2, 1])
    reg.SetSmoothingSigmasPerLevel([2, 1, 0])
    reg.SetInitialTransform(init, inPlace=False)
    final = reg.Execute(sitk.Cast(fixed_ct, sitk.sitkFloat32), sitk.Cast(moving_ct, sitk.sitkFloat32))
    return final, init


def onto(image, reference, transform, nearest=False):
    interp = sitk.sitkNearestNeighbor if nearest else sitk.sitkLinear
    return sitk.Resample(image, reference, transform, interp, 0.0, image.GetPixelID())


qc = []
for patient in PATIENTS:
    scans = data[patient]["scans"]
    ref = scans["scan1"]
    liver1 = ref["masks"]["organs_liver"]
    lesions1 = np.any([arr(m) > 0 for n, m in ref["masks"].items() if n.startswith("lesions_")], axis=0)
    data[patient]["liver_ref"] = {"liver_whole": liver1,
                                  "liver_healthy": mask_like((arr(liver1) > 0) & ~lesions1, liver1)}
    for scan, s in scans.items():
        if scan == "scan1":
            s["to_ref"] = sitk.Transform(3, sitk.sitkIdentity)
            s["spect_on_ref"] = s["spect"]
            continue
        final, init = register_liver(ref["ct"], s["ct"], liver1, s["masks"]["organs_liver"])
        s["to_ref"] = final
        s["spect_on_ref"] = onto(s["spect"], ref["spect"], final)
        ones = mask_like(np.ones(arr(s["spect"]).shape, bool), s["spect"])
        covered = arr(onto(ones, ref["spect"], final, nearest=True))[arr(liver1) > 0].mean()
        qc.append({"patient": patient, "scan": scan,
                   "dice_centroid_only": dice(liver1, onto(s["masks"]["organs_liver"], liver1, init, nearest=True)),
                   "dice_registered": dice(liver1, onto(s["masks"]["organs_liver"], liver1, final, nearest=True)),
                   "liver_coverage": covered,
                   "liver_activity_MBq": activity_mbq(s["spect_on_ref"], liver1)})
qc = pd.DataFrame(qc)
qc.round(3)

Visual check of the registration: coronal CT slice of scan 1 through the liver centroid, with the scan-1 liver contour (white) and the contours drawn on the other scans mapped to scan 1 (colours).

In [ ]:
colors = {"scan2": "tab:cyan", "scan3": "tab:orange", "scan4": "tab:red"}
fig, axes = plt.subplots(1, len(PATIENTS), figsize=(12, 6))
for ax, patient in zip(axes, PATIENTS):
    scans = data[patient]["scans"]
    liver1 = scans["scan1"]["masks"]["organs_liver"]
    y = int(round(np.argwhere(arr(liver1) > 0).mean(axis=0)[1]))
    ax.imshow(arr(scans["scan1"]["ct"])[::-1, y, :], cmap="gray", vmin=-160, vmax=240, aspect="auto")
    ax.contour(arr(liver1)[::-1, y, :], [0.5], colors="white", linewidths=1.5)
    for scan, c in colors.items():
        m = arr(onto(scans[scan]["masks"]["organs_liver"], liver1, scans[scan]["to_ref"], nearest=True))
        ax.contour(m[::-1, y, :], [0.5], colors=c, linewidths=1)
        ax.plot([], [], color=c, label=scan)
    ax.legend(loc="lower right", fontsize=8)
    ax.set_title(f"{patient}: scan-1 CT, liver contours mapped to scan 1")
    ax.axis("off")
fig.tight_layout()

## Visual check of lesion 2 of patient 6

Lesion 2 of patient 6 is small (about 4 mL, i.e. a few times the SPECT resolution volume) and its mean activity concentration rises by a factor of about 4 between scan 1 (3.7 h) and scan 2 (32.6 h), while the other lesions of the same patient are nearly flat over the same interval. Before fitting we check that the mask is on the lesion at every time point and that the rise is real and not a misplaced contour.

Top row: axial SPECT around the lesion with the lesion mask (cyan) and the liver mask of that scan (green), colour scale per scan. Middle row: CT. Bottom row: coronal SPECT. The title reports the mean concentration in the mask and the distance between the mask centroid and the hottest voxel within ±12 voxels.

In [ ]:
patient, lesion = "patient_6", "lesions_lesion2"
scans = data[patient]["scans"]
r, w = 25, 12
fig, axes = plt.subplots(3, len(scans), figsize=(4 * len(scans), 12))
for j, (scan, s) in enumerate(scans.items()):
    sp, ct = arr(s["spect"]), arr(s["ct"])
    m, liver = arr(s["masks"][lesion]) > 0, arr(s["masks"]["organs_liver"]) > 0
    z, y, x = np.round(np.argwhere(m).mean(axis=0)).astype(int)
    win = sp[z - w:z + w + 1, y - w:y + w + 1, x - w:x + w + 1]
    peak_mm = np.linalg.norm(np.array(np.unravel_index(win.argmax(), win.shape)) - w) * s["spect"].GetSpacing()[0]
    ax_sl = (z, slice(y - r, y + r), slice(x - r, x + r))
    co_sl = (slice(z - r, z + r), y, slice(x - r, x + r))
    vmax = sp[ax_sl].max()
    axes[0, j].imshow(sp[ax_sl], cmap="inferno", vmax=vmax)
    axes[0, j].contour(m[ax_sl], [0.5], colors="cyan")
    axes[0, j].contour(liver[ax_sl], [0.5], colors="lime", linewidths=0.6)
    axes[0, j].set_title(f"{scan}, t = {s['t_h']:.1f} h\nmean {sp[m].mean() / 1e6:.2f} MBq/mL, peak offset {peak_mm:.1f} mm")
    axes[1, j].imshow(ct[ax_sl], cmap="gray", vmin=-160, vmax=240)
    axes[1, j].contour(m[ax_sl], [0.5], colors="cyan")
    axes[1, j].contour(liver[ax_sl], [0.5], colors="lime", linewidths=0.6)
    axes[2, j].imshow(sp[co_sl][::-1], cmap="inferno", vmax=vmax)
    axes[2, j].contour(m[co_sl][::-1], [0.5], colors="cyan")
    for ax in axes[:, j]:
        ax.axis("off")
axes[1, 0].set_title("CT")
axes[2, 0].set_title("coronal SPECT")
fig.tight_layout()

conc = pd.DataFrame({
    name.removeprefix("lesions_"): [arr(s["spect"])[arr(s["masks"][name]) > 0].mean() / 1e6 for s in scans.values()]
    for name in scans["scan1"]["masks"] if name.startswith("lesions_")
}, index=[f"{k} ({v['t_h']:.1f} h)" for k, v in scans.items()])
conc.round(3).rename_axis("mean concentration [MBq/mL]")

**Outcome of the check.** At every time point the lesion-2 mask is centred on a focal hot spot inside the liver (hottest voxel within ~2 mm of the mask centroid), so the contours are correctly placed. The lesion is not visible on the non-contrast CT, which is why it was contoured on the SPECT. The increase between 3.7 h and 32.6 h is therefore a genuine **uptake phase** (typical of somatostatin-receptor binding, with the maximum around one day), followed by washout. Consequences for the fit:

- a mono-exponential model starting at $t = 0$ is not acceptable for this lesion; the uptake–washout model is required. The SNMMI analysis reported exactly this error for lesion 2 of patient B (mono-exponential fits used despite the visible uptake) and found the largest fitting variability for this lesion;
- with a volume of ~4 mL the partial-volume effect is large: the activity in the mask underestimates the true lesion activity. This affects the scale of the TIA more than its shape, and no recovery correction is applied here.

## Time-activity curves

- Kidneys, spleen and lesions: mask drawn on each scan, applied to that scan (as in the challenge).
- Liver (whole and healthy): scan-1 reference masks applied to every scan resampled on the scan-1 grid.
- `liver_whole_perscan`: liver with the mask of each scan, kept only to show the effect of the reference mask.

For every structure the scan-1 mask (or the liver reference) is also stored, for the comparison with the MIM map.

In [ ]:
structures = {}
tac = []
for patient in PATIENTS:
    scans = data[patient]["scans"]
    names = {n.split("_", 1)[1]: n for n in scans["scan1"]["masks"] if n != "organs_liver"}
    structures[patient] = {**{k: scans["scan1"]["masks"][v] for k, v in names.items()},
                           **data[patient]["liver_ref"],
                           "liver_whole_perscan": scans["scan1"]["masks"]["organs_liver"]}
    for scan, s in scans.items():
        row = {"patient": patient, "scan": scan, "t_h": s["t_h"]}
        for k, v in names.items():
            tac.append({**row, "structure": k, "activity_MBq": activity_mbq(s["spect"], s["masks"][v]),
                        "volume_ml": float((arr(s["masks"][v]) > 0).sum()) * voxel_ml(s["spect"])})
        for k, m in data[patient]["liver_ref"].items():
            tac.append({**row, "structure": k, "activity_MBq": activity_mbq(s["spect_on_ref"], m),
                        "volume_ml": float((arr(m) > 0).sum()) * voxel_ml(m)})
        tac.append({**row, "structure": "liver_whole_perscan",
                    "activity_MBq": activity_mbq(s["spect"], s["masks"]["organs_liver"]),
                    "volume_ml": float((arr(s["masks"]["organs_liver"]) > 0).sum()) * voxel_ml(s["spect"])})
tac = pd.DataFrame(tac)
tac.to_csv(OUT / "tac.csv", index=False)
tac.pivot_table(index=["patient", "structure"], columns="scan", values="activity_MBq").round(2)

In [ ]:
fig, axes = plt.subplots(1, len(PATIENTS), figsize=(13, 4.5), sharey=True)
for ax, patient in zip(axes, PATIENTS):
    for name, g in tac[tac.patient == patient].groupby("structure"):
        ls = ":" if name == "liver_whole_perscan" else "-"
        ax.plot(g.t_h, g.activity_MBq, "o" + ls, label=name, ms=4)
    ax.set_yscale("log")
    ax.set_xlabel("Time after injection [h]")
    ax.set_title(f"{patient} (SNMMI patient {SNMMI_LABEL[patient]})")
    ax.legend(fontsize=7, ncol=2)
axes[0].set_ylabel("Activity [MBq]")
fig.tight_layout()

## Kinetic models

Three nested models. In all of them the washout constant is $\lambda_w = \lambda_\text{phys} + \lambda_\text{bio}$ with $\lambda_\text{bio} \ge 0$: biological clearance can only add to physical decay, so after the uptake phase the activity of a region cannot decrease more slowly than physical decay.

| Model | $A(t)$ | Free parameters | TIA $= \int_0^\infty A\,dt$ |
|---|---|---|---|
| `phys` | $A_0 e^{-\lambda_\text{phys} t}$ | $A_0$ | $A_0/\lambda_\text{phys}$ |
| `mono` | $A_0 e^{-\lambda_w t}$ | $A_0, \lambda_\text{bio}$ | $A_0/\lambda_w$ |
| `uptake` | $A_0\left(e^{-\lambda_w t} - e^{-(\lambda_w + \lambda_u) t}\right)$ | $A_0, \lambda_\text{bio}, \lambda_u > 0$ | $A_0\left(\frac{1}{\lambda_w} - \frac{1}{\lambda_w+\lambda_u}\right)$ |

`mono` and `uptake` correspond to Eqs. 1 and 3 of Kurkowska et al. 2025; `uptake` has zero activity at injection and an uptake constant $\lambda_w + \lambda_u$ larger than the washout constant by construction, which removes the label-switching ambiguity of the usual bi-exponential. The 4-parameter bi-exponential (sum of two washout terms) is **not** used: with 4 time points it interpolates the data exactly and leaves no degree of freedom, as also stated in the SNMMI paper.

**Weighting.** Residuals are relative, $(A_\text{model} - A_i)/A_i$, i.e. a constant percentage uncertainty for every time point, which is closer to SPECT quantification errors than equal absolute weights and prevents the first, highest point from dominating the fit.

**One model for all structures: `uptake`.** The TIA of every organ and lesion comes from the uptake–washout model. `mono` and `phys` are still fitted, only as checks, and the AIC table is kept.

*Why model selection by AIC is not used.* With four points the small-sample correction $\text{AICc} = \text{AIC} + \frac{2k(k+1)}{n-k-1}$ is not defined for the 3-parameter model ($n - k - 1 = 0$), and plain AIC, $\text{AIC} = n \ln(\text{RSS}/n) + 2k$, tends to reward the more complex model, so a per-structure selection is unstable.

*Why the fixed rule "mono for organs, uptake for lesions" was dropped.* A previous version of this notebook fixed `mono` for kidneys, spleen and liver, following the SNMMI reference choice. For the **kidneys of patient 6** this rule fails: both curves have an uptake phase over the first day (the right kidney is higher at 32.6 h than at 3.7 h), the mono-exponential cannot follow it (relative residuals up to 24%), the AIC check flagged both curves ($\Delta\text{AIC} \approx 6$–7), and the kidney TIA fell about 10% below the MIM map, outside the 68% Monte Carlo interval; with `uptake` the ratio is 0.97–0.98. A rule that is right for one patient and wrong for the next cannot be used on a population whose kinetics are not known in advance.

*Why `uptake` is safe for organs as well.* `mono` is the limit of `uptake` for an instantaneous uptake ($\lambda_u \to \infty$). When the data show no uptake phase the fit returns a short uptake half-life and the TIA is the same as with `mono`: for the organs of patient 4 the difference is at most 2% (3% with the per-scan liver masks). When there is an uptake phase the model follows it. The cost is one degree of freedom less, which is accounted for by the Monte Carlo uncertainty below.

*Consistency with the rest of the project.* The same function is fitted voxel by voxel in notebook 03 and is the candidate physics layer of the network, so organ-level and voxel-level TIAs are computed with the same kinetic model.

**Checks.** For every structure the table reports the AIC of the three models, the model AIC would choose, and the ratio between the `mono` and the `uptake` TIA. A large ratio away from 1 marks a curve where the choice of model matters and that must be inspected in the figure.

**Extrapolation.** The fraction of the TIA after the last scan is reported; the EANM guidance recommends it to be below 20%.

In [ ]:
def a_phys(t, a0):
    return a0 * np.exp(-LAM_PHYS * t)


def a_mono(t, a0, lam_bio):
    return a0 * np.exp(-(LAM_PHYS + lam_bio) * t)


def a_uptake(t, a0, lam_bio, lam_up):
    lw = LAM_PHYS + lam_bio
    return a0 * (np.exp(-lw * t) - np.exp(-(lw + lam_up) * t))


def tia_from(model, p, t_from=0.0):
    lw = LAM_PHYS + p.get("lam_bio", 0.0)
    if model == "uptake":
        lu = lw + p["lam_up"]
        return p["a0"] * (np.exp(-lw * t_from) / lw - np.exp(-lu * t_from) / lu)
    return p["a0"] * np.exp(-lw * t_from) / lw


MODELS = {"phys": a_phys, "mono": a_mono, "uptake": a_uptake}
PARAM_NAMES = {"phys": ["a0"], "mono": ["a0", "lam_bio"], "uptake": ["a0", "lam_bio", "lam_up"]}
BOUNDS = {"phys": ([0.0], [np.inf]), "mono": ([0.0, 0.0], [np.inf, 2.0]),
          "uptake": ([0.0, 0.0, 1e-4], [np.inf, 2.0, 20.0])}


MODEL_USED = "uptake"


def starts(model, t, a):
    k = np.log(a[-2] / a[-1]) / (t[-1] - t[-2])
    lam_bio = max(k - LAM_PHYS, 1e-3)
    a0 = a[-1] * np.exp((LAM_PHYS + lam_bio) * t[-1])
    if model == "phys":
        return [{"a0": a[-1] * np.exp(LAM_PHYS * t[-1])}]
    if model == "mono":
        return [{"a0": a0, "lam_bio": lam_bio}]
    return [{"a0": a0, "lam_bio": lam_bio, "lam_up": np.log(2) / th} for th in (0.5, 2.0, 8.0, 24.0)]


def fit(model, t, a):
    m = lmfit.Model(MODELS[model])
    best = None
    for p0 in starts(model, t, a):
        params = m.make_params(**p0)
        for name, lo, hi in zip(PARAM_NAMES[model], *BOUNDS[model]):
            params[name].set(min=lo, max=hi)
        res = m.fit(a, params, t=t, weights=1.0 / a, nan_policy="raise")
        if best is None or res.chisqr < best.chisqr:
            best = res
    return best


def covariance_uncertainty(model, res):
    if res.covar is None:
        return np.nan
    p = {k: v.value for k, v in res.params.items()}
    g = np.zeros(len(res.var_names))
    for i, n in enumerate(res.var_names):
        q = dict(p)
        h = 1e-6 * max(abs(p[n]), 1e-8)
        q[n] += h
        g[i] = (tia_from(model, q) - tia_from(model, p)) / h
    return float(np.sqrt(g @ res.covar @ g))

In [ ]:
fits, fitted = [], {}
for (patient, name), g in tac.groupby(["patient", "structure"]):
    g = g.sort_values("t_h")
    t, a = g.t_h.to_numpy(), g.activity_MBq.to_numpy()
    results = {m: fit(m, t, a) for m in MODELS}
    aic = pd.Series({m: r.aic for m, r in results.items()})
    used = MODEL_USED
    fitted[(patient, name)] = (t, a, results, used)
    for m, r in results.items():
        p = {k: v.value for k, v in r.params.items()}
        tia = tia_from(m, p)
        fits.append({
            "patient": patient, "snmmi": SNMMI_LABEL[patient], "structure": name, "model": m,
            "used": m == used, "AIC": r.aic, "AIC_best_model": aic.idxmin(), "dAIC_vs_best": r.aic - aic.min(),
            "max_rel_residual": float(np.max(np.abs(r.residual))),
            "lam_bio_at_bound": m != "phys" and p["lam_bio"] < 1e-6,
            "T_eff_h": np.log(2) / (LAM_PHYS + p.get("lam_bio", 0.0)),
            "T_uptake_h": np.log(2) / (LAM_PHYS + p["lam_bio"] + p["lam_up"]) if m == "uptake" else np.nan,
            "TIA_MBq_h": tia, "TIA_cov_unc_MBq_h": covariance_uncertainty(m, r),
            "tail_fraction": tia_from(m, p, t[-1]) / tia,
        })
fits = pd.DataFrame(fits)
injected = {p: data[p]["timing"]["injected_Bq"] / 1e6 for p in PATIENTS}
fits["TIAC_h"] = fits.TIA_MBq_h / fits.patient.map(injected)
fits.to_csv(OUT / "fits_all_models.csv", index=False)

used = fits[fits.used].drop(columns="used").set_index(["patient", "structure"])
used[["snmmi", "model", "max_rel_residual", "lam_bio_at_bound", "T_eff_h", "T_uptake_h",
      "TIA_MBq_h", "TIAC_h", "tail_fraction"]].round(3)

### Checks: AIC and mono-exponential

`dAIC_used` is the AIC of the `uptake` model minus the lowest AIC among the three models (0 means AIC agrees). `TIA_mono_over_uptake` is the TIA of the mono-exponential check divided by the TIA used; values far from 1 identify curves with a real uptake phase (mono too low) or with an ill-constrained uptake fit.

In [ ]:
aic_table = fits.pivot_table(index=["patient", "structure"], columns="model", values="AIC")
aic_table["used"] = used.model
aic_table["AIC_best"] = used.AIC_best_model
aic_table["dAIC_used"] = used.dAIC_vs_best
aic_table["TIA_mono_over_uptake"] = (fits[fits.model == "mono"].set_index(["patient", "structure"]).TIA_MBq_h
                                     / used.TIA_MBq_h)
aic_table.round(3)

Fits: points = measured activity, solid line = model used, dashed = other models. The vertical line marks the last scan; everything to the right of it is extrapolation.

In [ ]:
styles = {"phys": "tab:gray", "mono": "tab:blue", "uptake": "tab:red"}
for patient in PATIENTS:
    keys = [k for k in fitted if k[0] == patient]
    ncol = 4
    nrow = int(np.ceil(len(keys) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(4 * ncol, 3.2 * nrow))
    for ax, key in zip(axes.ravel(), keys):
        t, a, results, model = fitted[key]
        tt = np.linspace(0, 1.6 * t[-1], 400)
        for m, r in results.items():
            p = {k: v.value for k, v in r.params.items()}
            ax.plot(tt, MODELS[m](tt, **p), color=styles[m], ls="-" if m == model else "--",
                    lw=2 if m == model else 1, label=m)
        ax.plot(t, a, "ko", ms=5)
        ax.axvline(t[-1], color="k", lw=0.5)
        ax.set_title(f"{key[1]}  [{model}; AIC best: {used.loc[key, 'AIC_best_model']}]", fontsize=9)
        ax.set_xlabel("t [h]")
        ax.set_ylabel("A [MBq]")
    for ax in axes.ravel()[len(keys):]:
        ax.axis("off")
    axes.ravel()[0].legend(fontsize=8)
    fig.suptitle(f"{patient} (SNMMI patient {SNMMI_LABEL[patient]})")
    fig.tight_layout()

## TIA uncertainty: Monte Carlo on the data points

The covariance matrix of the fit is not a usable uncertainty here: with 4 points and 3 parameters the `uptake` fit has a single degree of freedom, so the residual variance that scales the covariance is estimated from one number, and when a parameter sits on its bound ($\lambda_\text{bio} = 0$, effective half-life equal to the physical one) the linearised covariance is meaningless. The covariance values are kept in the table below only to show how optimistic they are.

Following the EANM guidance on uncertainty analysis for molecular radiotherapy dosimetry (Gear et al., *Eur J Nucl Med Mol Imaging* 2018;45:2456–2474), the uncertainty is propagated from the measured activities instead:

- **Random quantification error per time point**, independent between scans: each activity is multiplied by $1 + \sigma\,\varepsilon_i$, $\varepsilon_i \sim N(0,1)$. Two scenarios bracket the typical range for quantitative Lu-177 SPECT/CT: $\sigma = 5\%$ and $\sigma = 10\%$.
- **Partial volume (recovery) for lesions**, systematic: the same multiplicative factor applies to all time points of a lesion, so it scales the TIA directly. Assumed $\sigma_\text{PV} = 20\%$ for lesions below 10 mL and $10\%$ above. These are assumptions, to be replaced by a recovery curve measured on a phantom with the same reconstruction.
- A common calibration-factor uncertainty (a few percent, the same for every structure) is not included; it adds in quadrature to all results.

For every structure, $N = 1000$ replicas of the four activities are generated and refitted with the same model and bounds, starting from the nominal fit. Because the model and the relative weights are scale-invariant, the partial-volume factor multiplies the refitted TIA exactly and is applied after the fit. The result is summarised by the median and the 16th–84th percentiles; the relative uncertainty is half the 68% interval divided by the median. Bounded parameters are handled naturally: replicas whose best fit has $\lambda_\text{bio} = 0$ pile up at the bound and make the distribution asymmetric (`frac_at_bound`).

In [ ]:
from scipy.optimize import least_squares

SIGMA_POINT = {"5%": 0.05, "10%": 0.10}
N_MC = 1000


def sigma_pv(structure, volume_ml):
    if not structure.startswith("lesion"):
        return 0.0
    return 0.20 if volume_ml < 10.0 else 0.10


def refit(model, t, a, theta0):
    lb, ub = (np.array(b, dtype=float) for b in BOUNDS[model])
    f = MODELS[model]
    res = least_squares(lambda th: (f(t, *th) - a) / a, np.clip(theta0, lb + 1e-12, ub - 1e-12),
                        bounds=(lb, ub), method="trf", x_scale="jac")
    return dict(zip(PARAM_NAMES[model], res.x))


rng = np.random.default_rng(20261031)
mc, mc_samples = [], {}
for (patient, name), (t, a, results, model) in fitted.items():
    theta0 = np.array([results[model].params[n].value for n in PARAM_NAMES[model]])
    volume = tac[(tac.patient == patient) & (tac.structure == name) & (tac.scan == "scan1")].volume_ml.iloc[0]
    s_pv = sigma_pv(name, volume)
    nominal = used.loc[(patient, name), "TIA_MBq_h"]
    for label, s in SIGMA_POINT.items():
        eps = rng.standard_normal((N_MC, len(t)))
        tia_random, at_bound = np.empty(N_MC), 0
        for i in range(N_MC):
            p = refit(model, t, a * np.clip(1 + s * eps[i], 0.05, None), theta0)
            tia_random[i] = tia_from(model, p)
            at_bound += p.get("lam_bio", 1.0) < 1e-6
        tia_total = tia_random * np.clip(1 + s_pv * rng.standard_normal(N_MC), 0.05, None)
        r16, r50, r84 = np.percentile(tia_random, [16, 50, 84])
        q16, q50, q84 = np.percentile(tia_total, [16, 50, 84])
        mc_samples[(patient, name, label)] = tia_total
        mc.append({"patient": patient, "structure": name, "model": model, "sigma_point": label,
                   "sigma_pv": s_pv, "TIA_MBq_h": nominal, "TIA_median": q50, "TIA_p16": q16, "TIA_p84": q84,
                   "rel_unc_random_%": 100 * (r84 - r16) / 2 / r50,
                   "rel_unc_total_%": 100 * (q84 - q16) / 2 / q50,
                   "rel_unc_covariance_%": 100 * used.loc[(patient, name), "TIA_cov_unc_MBq_h"] / nominal,
                   "frac_at_bound": at_bound / N_MC})
mc = pd.DataFrame(mc)
mc.to_csv(OUT / "tia_uncertainty_mc.csv", index=False)

by = mc.set_index(["patient", "structure", "sigma_point"])
summary = by.xs("5%", level="sigma_point")[["model", "sigma_pv", "rel_unc_covariance_%"]].copy()
for label in SIGMA_POINT:
    sub = by.xs(label, level="sigma_point")
    summary[f"random_{label}"] = sub["rel_unc_random_%"]
    summary[f"total_{label}"] = sub["rel_unc_total_%"]
    summary[f"at_bound_{label}"] = sub["frac_at_bound"]
summary.round(2)

Distribution of the Monte Carlo TIA divided by the nominal TIA ($\sigma = 10\%$ scenario, partial volume included for lesions). Boxes span the 16th–84th percentiles, whiskers the 2.5th–97.5th.

In [ ]:
fig, axes = plt.subplots(1, len(PATIENTS), figsize=(13, 5), sharex=True)
for ax, patient in zip(axes, PATIENTS):
    names = [n for (p, n, l) in mc_samples if p == patient and l == "10%"]
    samples = [mc_samples[(patient, n, "10%")] / used.loc[(patient, n), "TIA_MBq_h"] for n in names]
    stats = [{"label": n, "med": np.median(x), "q1": np.percentile(x, 16), "q3": np.percentile(x, 84),
              "whislo": np.percentile(x, 2.5), "whishi": np.percentile(x, 97.5), "fliers": []}
             for n, x in zip(names, samples)]
    ax.bxp(stats, orientation="horizontal", showfliers=False)
    ax.axvline(1.0, color="k", lw=0.5)
    ax.set_xlabel("TIA / nominal TIA")
    ax.set_title(f"{patient} (SNMMI patient {SNMMI_LABEL[patient]})")
fig.tight_layout()

## Comparison with the MIM TIA map

The MIM map was built with voxel-wise fits after registering all scans to scan 1, with the model chosen per voxel by AIC. It is summed inside the scan-1 mask of each structure (inside the reference masks for the liver). Differences are expected from:
- voxel-wise vs organ-level fitting (the voxel-wise TIA is the sum of many noisy fits, and misaligned voxels at the organ edges lose or gain activity);
- the model set and the early-phase assumption used by MIM, which are not documented in the dataset;
- partial volume and registration for small lesions.

The 68% interval of the Monte Carlo ($\sigma = 10\%$) is reported next to the ratio; `MIM_in_68%` tells whether the MIM value is compatible with our estimate within that interval.

The kidneys of patient 6, which fell about 10% below MIM with the mono-exponential, are now within the interval (see the model section for why the mono rule was dropped).

In [ ]:
mim = []
for patient in PATIENTS:
    tia = data[patient]["tia"]
    for name, m in structures[patient].items():
        mim.append({"patient": patient, "structure": name, "TIA_MIM_MBq_h": activity_mbq(tia, m) / 3600.0})
mim = pd.DataFrame(mim).set_index(["patient", "structure"])

mc10 = mc[mc.sigma_point == "10%"].set_index(["patient", "structure"])
comparison = used[["model", "TIA_MBq_h"]].join(mc10[["TIA_p16", "TIA_p84", "rel_unc_total_%"]]).join(mim)
comparison["ratio_fit_over_MIM"] = comparison.TIA_MBq_h / comparison.TIA_MIM_MBq_h
comparison["MIM_in_68%"] = comparison.TIA_MIM_MBq_h.between(comparison.TIA_p16, comparison.TIA_p84)
comparison.to_csv(OUT / "tia_comparison_mim.csv")
comparison.round(3)

## Extrapolation after the last scan: a limitation of the protocol

For patient 4 the last scan is at 124 h, less than one physical half-life (159.5 h) after injection. For lesions, liver and spleen of this patient the fraction of the TIA extrapolated after the last scan is above the 20% recommended by EANM. This is not a fitting error but a limitation of the imaging protocol (last scan at day 5) and is declared as such: the TIA of these structures depends on an assumption about the washout after 124 h that the data cannot test. Patient 6, with the last scan at 193 h, stays at or below about 20%.

The extreme case is **lesion 2 of patient 4**: between the last two scans the activity decreases at the rate of physical decay, so the fit puts $\lambda_\text{bio}$ on its lower bound ($T_\text{eff} = T_\text{phys}$) and about 60% of the TIA comes from the extrapolation. Any biological washout after 124 h, which MIM's voxel fits partly include, lowers the TIA; this explains most of the ~40% difference from the MIM value. The Monte Carlo shows the same thing from the other side: a large fraction of the replicas end on the bound, and the distribution is strongly asymmetric.

This is exactly the uncertainty that the test-time optimisation of the project has to handle: a late, single post-therapy measurement constrains the washout, and the prediction from the pre-therapy PET must supply what the measurements do not.

In [ ]:
tails = used.reset_index()[["patient", "structure", "model", "T_eff_h", "lam_bio_at_bound", "tail_fraction"]]
tails["above_20%"] = tails.tail_fraction > 0.2
tails = tails.merge(mc10.reset_index()[["patient", "structure", "frac_at_bound"]], on=["patient", "structure"])
tails.round(3)

## Comparison with the SNMMI challenge

Per-patient TIA values of the participants are reported only graphically in the challenge papers, so the comparison is made on the quantitative findings that can be reproduced with these data (Kurkowska et al., *J Nucl Med* 2025;66:441–448):

1. **Decay correction.** Patient A images are decay-corrected to the injection time, patient B images to the start of the acquisition; ignoring this caused TIA overestimations of 43–204% among participants. Handled in notebook 01 and verified there against the MIM map.
2. **Hepatic lesions.** With the uptake–washout model (their Eq. 3), the liver TIA was **93.4% larger** for patient A and **6% larger** for patient B when lesions were included in the liver. Reproduced below with the scan-1 reference masks, with the uptake model (as in the paper and as used here) and, as a check, with `mono`.
3. **Fitting functions.** Mono-exponential for kidneys and the uptake–washout function for lesions were the reference choices; the 4-parameter bi-exponential is not recommended with 4 points; a decay constant smaller than the physical one should not be allowed. This notebook follows the last two rules and departs from the first: `uptake` is used for the kidneys too, because the kidneys of patient B show an uptake phase that the mono-exponential misses, and because `uptake` reduces to `mono` when the uptake is fast.
4. **Extrapolation.** The tail after the last scan was the largest source of TIA variability, especially for patient A (last scan at 124 h), consistent with the previous section.

In [ ]:
snmmi_liver = {"patient_4": 93.4, "patient_6": 6.0}
rows = []
for patient in PATIENTS:
    f = fits[fits.patient == patient].set_index(["structure", "model"])
    rows.append({
        "patient": patient, "snmmi": SNMMI_LABEL[patient],
        "increase_uptake_model_%": 100 * (f.loc[("liver_whole", "uptake"), "TIA_MBq_h"]
                                          / f.loc[("liver_healthy", "uptake"), "TIA_MBq_h"] - 1),
        "increase_mono_model_%": 100 * (f.loc[("liver_whole", "mono"), "TIA_MBq_h"]
                                        / f.loc[("liver_healthy", "mono"), "TIA_MBq_h"] - 1),
        "increase_MIM_%": 100 * (mim.loc[(patient, "liver_whole"), "TIA_MIM_MBq_h"]
                                 / mim.loc[(patient, "liver_healthy"), "TIA_MIM_MBq_h"] - 1),
        "SNMMI_published_%": snmmi_liver[patient],
    })
pd.DataFrame(rows).round(1)

## Saved files

In `data/processed/deepblue/kinetics/`:
- `tac.csv`: activity and volume per structure and time point;
- `fits_all_models.csv`: parameters, AIC, TIA, TIAC, tail fraction for every model (`used` = `uptake`);
- `tia_uncertainty_mc.csv`: Monte Carlo TIA uncertainty for both per-point scenarios (relative uncertainties in %);
- `tia_comparison_mim.csv`: `uptake` TIA, 68% interval and MIM map.

Notebook 03 moves the same models to the voxel level; the `uptake` model with $\lambda_\text{bio} \ge 0$ is the candidate physics layer for the network.